# Sentiment Analysis with a Bag-of-Words Network
We predict whether a movie review is positive or negative. Each review is turned into a *bag of words*: a vector with one entry per vocabulary word that counts how often the word appears. A fully connected network classifies that vector. This is the same task as `sentiment-network/`, built with PyTorch.

Cells marked **TODO** are yours to fill in; `Sentiment_Analysis_PyTorch_Solution.ipynb` has a complete version.

In [ ]:
from collections import Counter

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)

## Load the reviews
The text is already lower-cased and cleaned of most punctuation, so splitting on spaces is good enough.

In [ ]:
with open('../sentiment-network/reviews.txt') as f:
    reviews = f.read().split('\n')[:25000]
with open('../sentiment-network/labels.txt') as f:
    labels = f.read().split('\n')[:25000]

print(len(reviews), "reviews")
print(labels[0], "->", reviews[0][:150])

## Build the vocabulary
Count how often every word appears, and keep the 2,000 most common words. Rare words don't help the network much and make the input vectors huge.

**TODO:** fill `word2index`, a dictionary mapping each of the `vocab_size` most common words to a unique index `0 ... vocab_size-1`. (Hint: `Counter.most_common(n)` returns `(word, count)` pairs.)

In [ ]:
word_counts = Counter(word for review in reviews for word in review.split(' '))
vocab_size = 2000

# TODO: map each of the vocab_size most common words to an index
word2index = {}
print(len(word2index), "words in the vocabulary")

## Turn a review into a vector
**TODO:** write `text_to_vector(text)`. It returns a `float32` NumPy array of length `vocab_size` where entry `word2index[word]` counts the occurrences of `word` in the text. Words that are not in the vocabulary are ignored.

In [ ]:
def text_to_vector(text):
    # TODO: count the vocabulary words in the text
    pass

text_to_vector('the movie was great')[:10]

## Vectorize and split the data
Positive reviews get the label 1 and negative reviews 0. We shuffle the data, and keep the last 10% for testing.

In [ ]:
X = np.stack([text_to_vector(review) for review in reviews])
y = np.array([1.0 if label == 'positive' else 0.0 for label in labels], dtype=np.float32)

order = np.random.permutation(len(X))
split = int(len(X) * 0.9)
train_idx, test_idx = order[:split], order[split:]

X_train, y_train = torch.tensor(X[train_idx]), torch.tensor(y[train_idx])
X_test, y_test = torch.tensor(X[test_idx]), torch.tensor(y[test_idx])
print(X_train.shape, X_test.shape)

## Build the network
A hidden layer with dropout and a single output score. A positive score means "positive review"; `nn.BCEWithLogitsLoss` applies the sigmoid inside the loss.

**TODO:** define `model`, `criterion` and `optimizer`.

In [ ]:
# TODO: a network with vocab_size inputs and a single output score
model = None
criterion = nn.BCEWithLogitsLoss()
optimizer = None

## Train the network

In [ ]:
loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)

for epoch in range(5):
    model.train()
    total = 0.0
    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        loss = criterion(model(X_batch).squeeze(1), y_batch)
        loss.backward()
        optimizer.step()
        total += loss.item()
    print("Epoch %d  mean training loss %.4f" % (epoch + 1, total / len(loader)))

## Test the network
Anything above 80% is a decent result for a bag-of-words model.

In [ ]:
model.eval()
with torch.no_grad():
    predictions = (model(X_test).squeeze(1) > 0).float()
test_accuracy = (predictions == y_test).float().mean().item()
print("Test accuracy: %.4f" % test_accuracy)
assert test_accuracy > 0.8, "Try changing the network or the vocabulary size"

## Try your own review

In [ ]:
def predict(text):
    model.eval()
    with torch.no_grad():
        score = model(torch.tensor(text_to_vector(text.lower())).unsqueeze(0)).item()
    return ("positive" if score > 0 else "negative"), torch.sigmoid(torch.tensor(score)).item()

print(predict("this movie was great and the acting was wonderful"))
print(predict("a boring waste of time with terrible acting"))